In [1]:

import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import (f1_score, accuracy_score, classification_report,
                              confusion_matrix, precision_score, recall_score,
                              make_scorer)
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier


TRAIN_PATH = '/content/drive/MyDrive/finance project/customer_credit_train_clean.csv'
df = pd.read_csv(TRAIN_PATH)
UNSEEN_PATH = '/content/drive/MyDrive/finance project/customer_credit_unseen_clean.csv'
TARGET = 'Approved_Flag'
ID_COL = 'PROSPECTID'




In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# pip install catboost


================================================================================
PHASE 2: DATA UNDERSTANDING / EDA
================================================================================


In [4]:
def run_eda(df):
    print("=" * 60)
    print("PHASE 2: EDA")
    print("=" * 60)

    print("\nShape:", df.shape)

    print("\n--- Target distribution ---")
    print(df[TARGET].value_counts())
    print(df[TARGET].value_counts(normalize=True).round(3) * 100)


Finding: P2=62.7%, P3=14.5%, P4=11.5%, P1=11.3% -> real class imbalance.
A naive "always predict P2" model scores 62.7% accuracy while being
useless -- this is why Macro-F1 is used, not accuracy, throughout.


In [5]:

    print("\n--- Duplicate check ---")
    print("Duplicate PROSPECTID:", df[ID_COL].duplicated().sum())
    # Finding: 0 duplicates -> join between internal/external data was clean.

    print("\n--- Missing values (top 10) ---")
    print((df.isna().mean() * 100).sort_values(ascending=False).head(10))



--- Duplicate check ---
Duplicate PROSPECTID: 0

--- Missing values (top 10) ---
CC_utilization                  92.792582
PL_utilization                  86.557192
time_since_recent_deliquency    70.026882
time_since_first_deliquency     70.026882
max_delinquency_level           70.026882
max_unsec_exposure_inPct        45.149603
max_deliq_6mts                  25.109085
max_deliq_12mts                 21.100203
tot_enq                         12.312997
PL_enq                          12.312997
dtype: float64


Finding: CC_utilization (93%) / PL_utilization (87%) missing -- but this
is STRUCTURAL missingness (customer doesn't hold that product), not
random. Confirmed via has_credit_card / has_personal_loan flags.


In [6]:

    print("\n--- Key features by risk tier (median) ---")
    key_feats = ['NETMONTHLYINCOME', 'Credit_Score', 'Total_TL',
                 'num_times_delinquent', 'CC_utilization', 'tot_enq']
    for f in key_feats:
        if f in df.columns:
            print(f"\n{f}:")
            print(df.groupby(TARGET)[f].median())



--- Key features by risk tier (median) ---

NETMONTHLYINCOME:
Approved_Flag
P1    25000.0
P2    23000.0
P3    23000.0
P4    24715.0
Name: NETMONTHLYINCOME, dtype: float64

Credit_Score:
Approved_Flag
P1    712.0
P2    682.0
P3    665.0
P4    650.0
Name: Credit_Score, dtype: float64

Total_TL:
Approved_Flag
P1    6.0
P2    2.0
P3    2.0
P4    3.0
Name: Total_TL, dtype: float64

num_times_delinquent:
Approved_Flag
P1    0.0
P2    0.0
P3    0.0
P4    1.0
Name: num_times_delinquent, dtype: float64

CC_utilization:
Approved_Flag
P1    0.5030
P2    0.7110
P3    0.8230
P4    0.8965
Name: CC_utilization, dtype: float64

tot_enq:
Approved_Flag
P1    3.0
P2    3.0
P3    4.0
P4    7.0
Name: tot_enq, dtype: float64


Finding: Credit_Score decreases monotonically P1->P4 (712->650).
Utilization and enquiry count increase monotonically P1->P4.
Income shows weak separation -- behavior matters more than income.


In [7]:

    print("\n--- Outlier check on AGE ---")
    print(df['AGE'].describe())
    print("AGE <18 or >100:", ((df['AGE'] < 18) | (df['AGE'] > 100)).sum())
    # Finding: AGE range 21-77, no impossible values, no treatment needed.

    print("\n--- Correlation with target (ordinal encoding, quick signal check) ---")
    num_df = df.select_dtypes(include='number').copy()
    target_map = {'P1': 1, 'P2': 2, 'P3': 3, 'P4': 4}
    num_df['target_ord'] = df[TARGET].map(target_map)
    corr = num_df.corr()['target_ord'].abs().sort_values(ascending=False)
    print(corr.head(10))



--- Outlier check on AGE ---
count    51336.000000
mean        33.758532
std          8.816364
min         21.000000
25%         27.000000
50%         32.000000
75%         39.000000
max         77.000000
Name: AGE, dtype: float64
AGE <18 or >100: 0

--- Correlation with target (ordinal encoding, quick signal check) ---
target_ord                1.000000
Credit_Score              0.835461
enq_L3m                   0.538541
enq_L6m                   0.481929
enq_L12m                  0.425894
pct_PL_enq_L6m_of_ever    0.403911
pct_PL_enq_L6m_of_L12m    0.395561
Age_Oldest_TL             0.387620
PL_enq_L6m                0.357126
num_std_12mts             0.318561
Name: target_ord, dtype: float64


Finding: Credit_Score dominates (0.84). Recent enquiry counts (enq_L3m,
enq_L6m) are the next strongest signals -- "credit-seeking behavior"
is a well-known real-world risk marker.


In [8]:

    print("\n--- LEAKAGE / THRESHOLD CHECK: Credit_Score range per tier ---")
    print(df.groupby(TARGET)['Credit_Score'].agg(['min', 'max', 'mean', 'count']))



--- LEAKAGE / THRESHOLD CHECK: Credit_Score range per tier ---
               min  max        mean  count
Approved_Flag                             
P1             701  811  715.952266   5803
P2             669  700  682.584614  32199
P3             489  776  666.995035   7452
P4             469  658  645.629548   5882


CRITICAL FINDING: P1 range (701-811) and P2 range (669-700) do NOT
overlap -- a hard cutoff at exactly 700/701 suggests the bank's
original policy auto-assigns P1 above this threshold. P3/P4 ranges
overlap heavily with each other, meaning those tiers depend on more
than Credit_Score alone. This explains why models score ~99% WITH
Credit_Score, but drop to ~70-75% Macro-F1 WITHOUT it (tested
separately) -- the model is partly learning a known policy rule,
not pure novel prediction. Important to disclose, not leakage in
the technical sense since Credit_Score is legitimately available
at application time.


================================================================================
PHASE 3: FEATURE ENGINEERING
================================================================================


Only creates a feature if the underlying raw column exists --
    this keeps the function safe to reuse on the Unseen dataset, which
    has a smaller schema (42 cols) than Train (90 cols).


In [9]:
def engineer_features(d):
    d = d.copy()
    has = lambda c: c in d.columns

    if has('AGE'):
        d['age_group'] = pd.cut(d['AGE'], bins=[0, 25, 35, 45, 60, 100],
                                labels=['<25', '25-35', '35-45', '45-60', '60+'])

    if has('NETMONTHLYINCOME'):
        d['income_group'] = pd.qcut(d['NETMONTHLYINCOME'].rank(method='first'),
                                     q=4, labels=['Low', 'Mid-Low', 'Mid-High', 'High'])

    if has('Age_Oldest_TL'):
        d['credit_history_length'] = d['Age_Oldest_TL']

    cc_util = d['CC_utilization'].fillna(0) if has('CC_utilization') else 0
    pl_util = d['PL_utilization'].fillna(0) if has('PL_utilization') else 0

    if has('CC_utilization') or has('PL_utilization'):
        if has('CC_utilization') and has('PL_utilization'):
            d['avg_utilization'] = (cc_util + pl_util) / 2
        else:
            d['avg_utilization'] = cc_util if has('CC_utilization') else pl_util

        d['utilization_bucket'] = pd.cut(
            d['avg_utilization'], bins=[-0.01, 0.3, 0.6, 0.8, 1.01],
            labels=['Low', 'Moderate', 'High', 'Very High']
        )

    delinq_cols = [c for c in ['num_times_delinquent', 'num_times_30p_dpd', 'num_times_60p_dpd'] if has(c)]
    if delinq_cols:
        d['total_delinquency_score'] = d[delinq_cols].sum(axis=1)

    if has('Tot_Active_TL') and has('Total_TL'):
        d['active_loan_count'] = d['Tot_Active_TL']
        d['loan_burden_ratio'] = (d['Tot_Active_TL'] / d['Total_TL'].replace(0, np.nan)).fillna(0)

    product_cols = [c for c in ['Auto_TL', 'CC_TL', 'Consumer_TL', 'Gold_TL', 'Home_TL', 'PL_TL'] if has(c)]
    if product_cols:
        d['credit_mix_count'] = (d[product_cols] > 0).sum(axis=1)

    if has('enq_L3m') and has('enq_L12m'):
        d['recent_enquiry_intensity'] = (d['enq_L3m'] / d['enq_L12m'].replace(0, np.nan)).fillna(0)

    return d

NOTE: fillna(0) here is deliberate -- missing utilization means
"no credit card/personal loan held", NOT "unknown usage".
Filling with mean would fabricate fake card usage for people
who never had a card.


Share of ever-held loans still active -- high ratio = still
juggling most obligations ever taken on.


Diversity of credit products held -- generally a lower-risk signal
in bureau scoring models.


Share of all enquiries concentrated very recently -- a spike here
signals urgent/desperate credit-seeking, a known risk flag.


================================================================================
PHASE 4: FEATURE SELECTION
================================================================================


In [10]:
def select_features(df, target=TARGET, id_col=ID_COL):
    drop_cols = [id_col, target]
    cat_cols = [c for c in df.select_dtypes(include=['object', 'category']).columns
                if c not in drop_cols]

    X = df.drop(columns=drop_cols).copy()
    for c in cat_cols:
        X[c] = LabelEncoder().fit_transform(X[c].astype(str))
    X_temp = X.fillna(-1)  # temporary fill, ONLY for selection statistics
    y = LabelEncoder().fit_transform(df[target])

    # --- Multicollinearity: flag pairs with |corr| > 0.9 ---
    corr_matrix = X_temp.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    high_corr_pairs = [(col, row, upper.loc[row, col])
                        for col in upper.columns for row in upper.index
                        if upper.loc[row, col] > 0.9]
    print(f"Highly correlated pairs (>0.9): {len(high_corr_pairs)}")

    # --- Near-zero variance columns ---
    variances = X_temp.var()
    low_var_cols = variances[variances < 0.01].index.tolist()
    print(f"Near-zero variance columns: {low_var_cols}")

    # --- Random Forest importance ---
    rf = RandomForestClassifier(n_estimators=200, random_state=42,
                                 n_jobs=-1, max_depth=10)
    rf.fit(X_temp, y)
    importances = pd.Series(rf.feature_importances_,
                             index=X_temp.columns).sort_values(ascending=False)
    print("\nTop 10 RF importances:\n", importances.head(10))

    # --- Mutual Information ---
    mi = mutual_info_classif(X_temp, y, random_state=42, n_neighbors=3)
    mi_series = pd.Series(mi, index=X_temp.columns).sort_values(ascending=False)
    print("\nTop 10 Mutual Information scores:\n", mi_series.head(10))

    # Decision: drop near-zero-variance cols, and the more redundant half
    # of each near-duplicate pair (keep the more granular/recent version).
    drop_redundant = [
        'Tot_Closed_TL', 'pct_closed_tl', 'pct_of_active_TLs_ever', 'Secured_TL',
        'num_deliq_12mts', 'num_times_60p_dpd', 'num_std_12mts', 'num_dbt_12mts',
        'num_lss_12mts', 'CC_enq_L12m', 'PL_enq_L12m', 'enq_L12m',
        'pct_PL_enq_L6m_of_L12m',
    ]
    all_drops = [c for c in (low_var_cols + drop_redundant) if c in df.columns]
    df_selected = df.drop(columns=all_drops)
    print(f"\nDropped {len(all_drops)} columns. Remaining: {df_selected.shape[1]}")
    return df_selected

================================================================================
PHASE 5: DATA PREPARATION
================================================================================


In [11]:
def prepare_data(df, target=TARGET, id_col=ID_COL):
    X = df.drop(columns=[target, id_col])
    y = df[target]

    # Structural-zero imputation done BEFORE the pipeline split, since it
    # reflects a business fact (no product held), not statistical noise.
    for c in ['CC_utilization', 'PL_utilization']:
        if c in X.columns:
            X[c] = X[c].fillna(0)

    # Delinquency-timing columns: missing = "never delinquent". Use a
    # sentinel (-1) distinct from any real time value, not median/mean,
    # which would fabricate a fake delinquency date.
    for c in [col for col in X.columns if 'time_since' in col and 'deliq' in col]:
        X[c] = X[c].fillna(-1)

    cat_cols = X.select_dtypes(include=['object']).columns.tolist()
    num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

    # Stratified 70/15/15 split -- preserves the P1/P2/P3/P4 ratio in
    # every split so no subset over/under-represents a minority class.
    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y, test_size=0.15, stratify=y, random_state=42)
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=0.176, stratify=y_temp, random_state=42)

    # Preprocessing pipeline fit ONLY on training data -- prevents
    # validation/test distribution info from leaking into training.
    preprocessor = ColumnTransformer([
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), num_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), cat_cols)
    ])
    preprocessor.fit(X_train)

    le = LabelEncoder()
    y_train_enc = le.fit_transform(y_train)
    y_val_enc = le.transform(y_val)
    y_test_enc = le.transform(y_test)

    print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
    print("Class balance preserved across splits:")
    print(pd.Series(y_train).value_counts(normalize=True).round(3))

    return (X_train, X_val, X_test, y_train_enc, y_val_enc, y_test_enc,
            preprocessor, le)


================================================================================
PHASE 6: MODEL BUILDING
================================================================================


In [12]:
def train_all_models(X_train_t, y_train_enc, X_val_t, y_val_enc):
    """Train 7 candidate models. Model selection uses Macro-F1, not
    accuracy, because accuracy is misleading under class imbalance
    (P2 = 62.7% of the data)."""
    models = {
        'Logistic_Regression': LogisticRegression(
            max_iter=500, class_weight='balanced', random_state=42),
        'Decision_Tree': DecisionTreeClassifier(
            class_weight='balanced', max_depth=10, random_state=42),
        'Random_Forest': RandomForestClassifier(
            n_estimators=200, class_weight='balanced', max_depth=15,
            random_state=42, n_jobs=-1),
        'Gradient_Boosting': GradientBoostingClassifier(
            n_estimators=100, max_depth=4, random_state=42),
        'XGBoost': xgb.XGBClassifier(
            n_estimators=200, max_depth=6, learning_rate=0.1,
            objective='multi:softmax', num_class=4, random_state=42,
            n_jobs=-1, eval_metric='mlogloss'),
        'LightGBM': lgb.LGBMClassifier(
            n_estimators=200, max_depth=6, learning_rate=0.1,
            class_weight='balanced', random_state=42, verbose=-1),
        'CatBoost': CatBoostClassifier(
            iterations=200, depth=6, learning_rate=0.1, random_state=42,
            verbose=0, auto_class_weights='Balanced'),
    }

    results = {}
    for name, model in models.items():
        model.fit(X_train_t, y_train_enc)
        pred = model.predict(X_val_t)
        results[name] = {
            'model': model,
            'accuracy': accuracy_score(y_val_enc, pred),
            'f1_macro': f1_score(y_val_enc, pred, average='macro'),
            'f1_weighted': f1_score(y_val_enc, pred, average='weighted'),
        }
        print(f"{name}: Acc={results[name]['accuracy']:.4f}  "
              f"MacroF1={results[name]['f1_macro']:.4f}")
    # Result: all boosting models cluster near 99% Macro-F1; Logistic
    # Regression is weakest (89%) since the Credit_Score threshold effect
    # is non-linear. CatBoost edges out slightly and handles categoricals
    # natively -- chosen as final model.
    return results

================================================================================
PHASE 7: HYPERPARAMETER TUNING
================================================================================


In [13]:
def tune_xgboost(X_train_t, y_train_enc, X_val_t, y_val_enc):
    """RandomizedSearchCV (faster than GridSearch, sufficient given models
    are already near-ceiling performance). Optimizes Macro-F1."""
    param_dist = {
        'n_estimators': [100, 200, 300],
        'max_depth': [4, 6, 8],
        'learning_rate': [0.05, 0.1, 0.2],
        'subsample': [0.8, 1.0],
        'colsample_bytree': [0.8, 1.0],
    }
    macro_f1_scorer = make_scorer(f1_score, average='macro')
    base = xgb.XGBClassifier(objective='multi:softmax', num_class=4,
                              random_state=42, n_jobs=-1, eval_metric='mlogloss')
    search = RandomizedSearchCV(base, param_dist, n_iter=10,
                                 scoring=macro_f1_scorer, cv=3,
                                 random_state=42, n_jobs=-1)
    search.fit(X_train_t, y_train_enc)

    best_model = search.best_estimator_
    pred = best_model.predict(X_val_t)
    print("Best params:", search.best_params_)
    print(f"Tuned val Macro-F1: {f1_score(y_val_enc, pred, average='macro'):.4f}")
    # Result: tuning improved XGBoost Macro-F1 only marginally (99.08% ->
    # 99.13%) -- expected, since untuned models were already near-ceiling.
    return best_model




================================================================================
PHASE 8: FINAL MODEL EVALUATION (held-out test set)
================================================================================


In [14]:
def evaluate_final_model(model, X_test_t, y_test_enc, class_names):
    """The test set is touched here for the FIRST time -- never used
    during training, feature selection, or tuning, to get an unbiased
    estimate of real-world performance."""
    pred = model.predict(X_test_t).ravel()

    print("=== FINAL TEST SET PERFORMANCE ===")
    print(f"Accuracy:        {accuracy_score(y_test_enc, pred):.4f}")
    print(f"Macro F1:        {f1_score(y_test_enc, pred, average='macro'):.4f}")
    print(f"Weighted F1:     {f1_score(y_test_enc, pred, average='weighted'):.4f}")
    print(f"Macro Precision: {precision_score(y_test_enc, pred, average='macro'):.4f}")
    print(f"Macro Recall:    {recall_score(y_test_enc, pred, average='macro'):.4f}")

    print("\n", classification_report(y_test_enc, pred, target_names=class_names))

    cm = confusion_matrix(y_test_enc, pred)
    print("Confusion Matrix:\n",
          pd.DataFrame(cm, index=class_names, columns=class_names))
    # Result: Macro-F1 = 99.00% on test set, confirming the validation
    # result generalizes (not overfit to validation). Main confusion:
    # 34 P1 misclassified as P3 -- consistent with the overlapping
    # Credit_Score ranges found during EDA's leakage/threshold check.
    return pred, cm




In [16]:
# 1. Get the text labels for the true classes
true_labels = le.inverse_transform(y_test_enc)

# 2. Get the model's predictions (using the processed test features)
test_predictions_enc = final_model.predict(X_test_t).ravel()
predicted_labels = le.inverse_transform(test_predictions_enc)

# 3. Combine into a DataFrame to inspect
verification_df = pd.DataFrame({
    'Actual_Tier': true_labels,
    'Predicted_Tier': predicted_labels,
    'Correct_Prediction': true_labels == predicted_labels
})

# View the first 20 rows
print(verification_df.head(20))

# See total misclassifications
print(f"\nTotal Misclassified: {(verification_df['Correct_Prediction'] == False).sum()} out of {len(verification_df)}")

   Actual_Tier Predicted_Tier  Correct_Prediction
0           P4             P4                True
1           P2             P2                True
2           P3             P3                True
3           P2             P2                True
4           P2             P2                True
5           P1             P1                True
6           P4             P4                True
7           P4             P4                True
8           P3             P3                True
9           P2             P2                True
10          P2             P2                True
11          P1             P1                True
12          P2             P2                True
13          P2             P2                True
14          P2             P2                True
15          P2             P2                True
16          P2             P2                True
17          P2             P2                True
18          P3             P3                True


================================================================================
MAIN PIPELINE
================================================================================


In [15]:
if __name__ == '__main__':
    # Load
    df_raw = pd.read_csv(TRAIN_PATH)
    unseen_raw = pd.read_csv(UNSEEN_PATH)

    # Phase 2
    run_eda(df_raw)

    # Phase 3
    df_fe = engineer_features(df_raw)
    unseen_fe = engineer_features(unseen_raw)

    # Phase 4
    df_selected = select_features(df_fe)

    # Phase 5
    (X_train, X_val, X_test, y_train_enc, y_val_enc, y_test_enc,
     preprocessor, le) = prepare_data(df_selected)

    X_train_t = preprocessor.transform(X_train)
    X_val_t = preprocessor.transform(X_val)
    X_test_t = preprocessor.transform(X_test)

    # Phase 6
    results = train_all_models(X_train_t, y_train_enc, X_val_t, y_val_enc)
    best_name = max(results, key=lambda k: results[k]['f1_macro'])
    print(f"\nBest model by Macro-F1: {best_name}")

    # Phase 7
    tuned_xgb = tune_xgboost(X_train_t, y_train_enc, X_val_t, y_val_enc)

    # Phase 8 -- final chosen model is CatBoost (best untuned Macro-F1,
    # native categorical handling; CatBoost tuning was skipped after
    # timing out during search -- marginal gain expected given how close
    # all boosting models already were)
    final_model = results['CatBoost']['model']
    evaluate_final_model(final_model, X_test_t, y_test_enc, le.classes_)

    joblib.dump(final_model, 'final_model.pkl')
    joblib.dump(preprocessor, 'final_preprocessor.pkl')
    joblib.dump(le, 'final_label_encoder.pkl')

PHASE 2: EDA

Shape: (51336, 90)

--- Target distribution ---
Approved_Flag
P2    32199
P3     7452
P4     5882
P1     5803
Name: count, dtype: int64
Approved_Flag
P2    62.7
P3    14.5
P4    11.5
P1    11.3
Name: proportion, dtype: float64
Highly correlated pairs (>0.9): 25
Near-zero variance columns: ['num_sub_6mts', 'num_dbt_6mts', 'num_lss_6mts']

Top 10 RF importances:
 Credit_Score                0.467183
enq_L3m                     0.051348
Age_Oldest_TL               0.049752
credit_history_length       0.046644
enq_L6m                     0.032498
recent_enquiry_intensity    0.029762
num_std                     0.025512
num_std_12mts               0.025092
time_since_recent_enq       0.020071
pct_PL_enq_L6m_of_ever      0.016034
dtype: float64

Top 10 Mutual Information scores:
 Credit_Score                1.057671
enq_L3m                     0.179622
enq_L6m                     0.153315
recent_enquiry_intensity    0.148342
credit_history_length       0.140053
Age_Oldest_TL   

In [3]:
"""
PHASE 11: Prediction Pipeline (Deployment Model)
--------------------------------------------------
Constraint discovered: Unseen dataset has only 42 columns, missing
Credit_Score, AGE, and ~44 other columns present in Train. The Track A
model (99% Macro-F1, uses Credit_Score) CANNOT run on Unseen.
This builds Track B: trained ONLY on the 42 common columns, so it can
actually produce predictions on the real holdout set.
"""
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, classification_report
from catboost import CatBoostClassifier

TARGET = 'Approved_Flag'
ID_COL = 'PROSPECTID'

train_path = r'C:\Users\DURVA\Finance Project\customer_credit_train_clean.csv'
unseen_path = r'C:\Users\DURVA\Finance Project\customer_credit_unseen_clean.csv'

# 2. Read them into DataFrames
train = pd.read_csv(train_path)
unseen = pd.read_csv(unseen_path)

common_cols = sorted(set(train.columns) & set(unseen.columns))
print(f"Common feature columns: {len(common_cols)}")

X = train[common_cols].copy()
y = train[TARGET]

# same structural-missing handling as before, restricted to what's present here
if 'CC_utilization' in X.columns:
    X['CC_utilization'] = X['CC_utilization'].fillna(0)
if 'PL_utilization' in X.columns:
    X['PL_utilization'] = X['PL_utilization'].fillna(0)

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=42)

preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')),
                       ('scaler', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                       ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols)
])
preprocessor.fit(X_train)
X_train_t = preprocessor.transform(X_train)
X_test_t = preprocessor.transform(X_test)

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

model = CatBoostClassifier(iterations=250, depth=6, learning_rate=0.1,
                            random_state=42, verbose=0, auto_class_weights='Balanced')
model.fit(X_train_t, y_train_enc)

pred = model.predict(X_test_t).ravel()
acc = accuracy_score(y_test_enc, pred)
f1m = f1_score(y_test_enc, pred, average='macro')
print(f"\nDEPLOYMENT MODEL (42 features, no Credit_Score/AGE) test performance:")
print(f"Accuracy: {acc:.4f}   Macro-F1: {f1m:.4f}")
print(classification_report(y_test_enc, pred, target_names=le.classes_))

joblib.dump(model, 'deployment_model.pkl')
joblib.dump(preprocessor, 'deployment_preprocessor.pkl')
joblib.dump(le, 'deployment_label_encoder.pkl')
joblib.dump(common_cols, 'deployment_columns.pkl')

ModuleNotFoundError: No module named 'pandas'

In [21]:
"""
PHASE 11 (continued): Predict on the real Unseen dataset.
"""
import pandas as pd
import numpy as np
import joblib

model = joblib.load('deployment_model.pkl')
preprocessor = joblib.load('deployment_preprocessor.pkl')
le = joblib.load('deployment_label_encoder.pkl')
common_cols = joblib.load('deployment_columns.pkl')

X_unseen = unseen[common_cols].copy()
if 'CC_utilization' in X_unseen.columns:
    X_unseen['CC_utilization'] = X_unseen['CC_utilization'].fillna(0)
if 'PL_utilization' in X_unseen.columns:
    X_unseen['PL_utilization'] = X_unseen['PL_utilization'].fillna(0)

X_unseen_t = preprocessor.transform(X_unseen)

pred_enc = model.predict(X_unseen_t).ravel()
pred_proba = model.predict_proba(X_unseen_t)

pred_labels = le.inverse_transform(pred_enc)
confidence = pred_proba.max(axis=1)

def business_explanation(row_idx):
    tier = pred_labels[row_idx]
    conf = confidence[row_idx]
    if tier == 'P1':
        return f"Low risk (confidence {conf:.0%}) - candidate for fast-track approval, standard rate"
    elif tier == 'P2':
        return f"Moderate risk (confidence {conf:.0%}) - standard approval process, standard-to-slightly-elevated rate"
    elif tier == 'P3':
        return f"Elevated risk (confidence {conf:.0%}) - recommend additional verification before approval"
    else:  # P4
        return f"High risk (confidence {conf:.0%}) - recommend manual underwriting review, collateral or higher rate if approved"

results = pd.DataFrame({
    'Customer_Index': range(len(unseen)),
    'Predicted_Risk_Tier': pred_labels,
    'Confidence_Score': confidence.round(3),
    'Prob_P1': pred_proba[:, list(le.classes_).index('P1')].round(3),
    'Prob_P2': pred_proba[:, list(le.classes_).index('P2')].round(3),
    'Prob_P3': pred_proba[:, list(le.classes_).index('P3')].round(3),
    'Prob_P4': pred_proba[:, list(le.classes_).index('P4')].round(3),
    'Business_Recommendation': [business_explanation(i) for i in range(len(unseen))]
})

print("Predicted tier distribution on Unseen (100 customers):")
print(results['Predicted_Risk_Tier'].value_counts())
print()
print(results.head(15).to_string())

results.to_csv('customer_credit_predictions.csv', index=False)
print("\nSaved: customer_credit_predictions.csv")

Predicted tier distribution on Unseen (100 customers):
Predicted_Risk_Tier
P2    50
P3    22
P4    15
P1    13
Name: count, dtype: int64

    Customer_Index Predicted_Risk_Tier  Confidence_Score  Prob_P1  Prob_P2  Prob_P3  Prob_P4                                                                                   Business_Recommendation
0                0                  P2             0.410    0.324    0.410    0.242    0.024            Moderate risk (confidence 41%) - standard approval process, standard-to-slightly-elevated rate
1                1                  P2             0.948    0.002    0.948    0.048    0.002            Moderate risk (confidence 95%) - standard approval process, standard-to-slightly-elevated rate
2                2                  P2             0.747    0.116    0.747    0.126    0.011            Moderate risk (confidence 75%) - standard approval process, standard-to-slightly-elevated rate
3                3                  P1             0.995    0.995 

In [22]:
# Temporary merge to inspect features by predicted risk tier
check_df = unseen.copy()
check_df['Predicted_Tier'] = results['Predicted_Risk_Tier']

# Check if riskier tiers have higher inquiries or higher utilization
key_bureau_features = ['enq_L3m', 'enq_L6m', 'CC_utilization']
for feat in key_bureau_features:
    if feat in check_df.columns:
        print(f"\nMean {feat} by Predicted Tier:")
        print(check_df.groupby('Predicted_Tier')[feat].mean())


Mean enq_L3m by Predicted Tier:
Predicted_Tier
P1    0.692308
P2    0.220000
P3    1.318182
P4    4.000000
Name: enq_L3m, dtype: float64


In [25]:
# Check a bureau feature that exists in your deployment columns safely
available_feats = [col for col in ['Tot_Active_TL', 'Total_TL', 'enq_L3m'] if col in check_df.columns]

for feat in available_feats:
    print(f"\nMean {feat} by Predicted Tier:")
    print(check_df.groupby('Predicted_Tier')[feat].mean())


Mean enq_L3m by Predicted Tier:
Predicted_Tier
P1    0.692308
P2    0.220000
P3    1.318182
P4    4.000000
Name: enq_L3m, dtype: float64


In [26]:
# Print all 42 column names in a clean list
print("Available columns in unseen dataset:")
print(list(X_unseen.columns))

Available columns in unseen dataset:
['Age_Newest_TL', 'Age_Oldest_TL', 'CC_Flag', 'CC_TL', 'CC_enq_L12m', 'EDUCATION', 'GENDER', 'GL_Flag', 'HL_Flag', 'Home_TL', 'MARITALSTATUS', 'NETMONTHLYINCOME', 'Other_TL', 'PL_Flag', 'PL_TL', 'PL_enq_L12m', 'Secured_TL', 'Time_With_Curr_Empr', 'Tot_Missed_Pmnt', 'Tot_TL_closed_L12M', 'Unsecured_TL', 'enq_L3m', 'first_prod_enq2', 'last_prod_enq2', 'max_recent_level_of_deliq', 'num_dbt', 'num_dbt_12mts', 'num_deliq_6_12mts', 'num_lss', 'num_std_12mts', 'num_sub', 'num_sub_12mts', 'num_sub_6mts', 'num_times_60p_dpd', 'pct_CC_enq_L6m_of_ever', 'pct_PL_enq_L6m_of_ever', 'pct_tl_closed_L12M', 'pct_tl_closed_L6M', 'pct_tl_open_L6M', 'recent_level_of_deliq', 'time_since_recent_enq', 'time_since_recent_payment']


In [27]:
# Replace 'YOUR_COLUMN_HERE' with a valid column name from the list above
chosen_feat = 'PL_Flag'

print(f"\nMean {chosen_feat} by Predicted Tier:")
print(check_df.groupby('Predicted_Tier')[chosen_feat].mean())


Mean PL_Flag by Predicted Tier:
Predicted_Tier
P1    0.384615
P2    0.160000
P3    0.227273
P4    0.333333
Name: PL_Flag, dtype: float64


In [28]:
chosen_feat = 'Age_Oldest_TL'

print(f"\nMean {chosen_feat} by Predicted Tier:")
print(check_df.groupby('Predicted_Tier')[chosen_feat].mean())


Mean Age_Oldest_TL by Predicted Tier:
Predicted_Tier
P1    95.307692
P2    34.500000
P3    33.727273
P4    39.466667
Name: Age_Oldest_TL, dtype: float64


In [33]:
import shap
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

model = joblib.load('deployment_model.pkl')
preprocessor = joblib.load('deployment_preprocessor.pkl')
le = joblib.load('deployment_label_encoder.pkl')
common_cols = joblib.load('deployment_columns.pkl')

TRAIN_PATH = '/content/drive/MyDrive/finance project/customer_credit_train_clean.csv'
train = pd.read_csv(TRAIN_PATH)

X = train[common_cols].copy()
if 'CC_utilization' in X.columns:
    X['CC_utilization'] = X['CC_utilization'].fillna(0)
if 'PL_utilization' in X.columns:
    X['PL_utilization'] = X['PL_utilization'].fillna(0)

# use a sample for SHAP speed (standard practice on large datasets)
X_sample = X.sample(1000, random_state=42)
X_sample_t = preprocessor.transform(X_sample)

# get transformed feature names
num_cols = preprocessor.transformers_[0][2]
cat_cols = preprocessor.transformers_[1][2]
ohe = preprocessor.named_transformers_['cat'].named_steps['onehot']
cat_feature_names = ohe.get_feature_names_out(cat_cols).tolist()
all_feature_names = num_cols + cat_feature_names

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_sample_t)

print("SHAP values shape:", np.array(shap_values).shape)
print("Classes:", le.classes_)

# ---- 1. Summary plot (global feature importance across all classes) ----
# shap_values is (n_samples, n_features, n_classes) for multiclass CatBoost
shap.summary_plot(shap_values, X_sample_t, feature_names=all_feature_names,
                   class_names=le.classes_, show=False, max_display=15)
plt.tight_layout()
plt.savefig('summary_plot.png', dpi=100)
plt.close()
print("Saved summary_plot.png")

# ---- 2. Feature importance (mean abs SHAP across classes) ----
mean_abs_shap = np.abs(shap_values).mean(axis=(0, 2))
importance_df = pd.DataFrame({'feature': all_feature_names, 'mean_abs_shap': mean_abs_shap})
importance_df = importance_df.sort_values('mean_abs_shap', ascending=False)
print("\nTop 15 features by mean |SHAP|:")
print(importance_df.head(15).to_string())
importance_df.to_csv('shap_feature_importance.csv', index=False)

joblib.dump(explainer, 'shap_explainer.pkl')
np.save('shap_values_sample.npy', shap_values)
X_sample.to_csv('shap_X_sample.csv', index=False)
joblib.dump(all_feature_names, 'shap_feature_names.pkl')


SHAP values shape: (1000, 60, 4)
Classes: ['P1' 'P2' 'P3' 'P4']
Saved summary_plot.png

Top 15 features by mean |SHAP|:
                      feature  mean_abs_shap
1               Age_Oldest_TL       0.797882
18                    enq_L3m       0.555423
24              num_std_12mts       0.437318
35      time_since_recent_enq       0.395316
19  max_recent_level_of_deliq       0.337506
30     pct_PL_enq_L6m_of_ever       0.329197
34      recent_level_of_deliq       0.269286
0               Age_Newest_TL       0.105289
22          num_deliq_6_12mts       0.049362
36  time_since_recent_payment       0.048714
12                PL_enq_L12m       0.043309
33            pct_tl_open_L6M       0.041903
28          num_times_60p_dpd       0.037283
9                    Other_TL       0.033844
8            NETMONTHLYINCOME       0.031255


['shap_feature_names.pkl']

In [34]:
shap_values = np.load('shap_values_sample.npy')
X_sample = pd.read_csv('shap_X_sample.csv')
feature_names = joblib.load('shap_feature_names.pkl')
le = joblib.load('deployment_label_encoder.pkl')
model = joblib.load('deployment_model.pkl')
preprocessor = joblib.load('deployment_preprocessor.pkl')

X_sample_t = preprocessor.transform(X_sample)
preds = model.predict(X_sample_t).ravel()

# ---- Waterfall plot for one P4 (high risk) customer ----
p4_idx = np.where(preds == list(le.classes_).index('P4'))[0]
if len(p4_idx) > 0:
    idx = p4_idx[0]
    class_idx = list(le.classes_).index('P4')
    explanation = shap.Explanation(
        values=shap_values[idx, :, class_idx],
        base_values=0,  # CatBoost multiclass base handled internally; using relative view
        data=X_sample_t[idx],
        feature_names=feature_names
    )
    shap.plots.waterfall(explanation, max_display=12, show=False)
    plt.tight_layout()
    plt.savefig('waterfall_P4_customer.png', dpi=100)
    plt.close()
    print(f"Waterfall saved for customer index {idx}, predicted P4")
    print("Their key raw values:")
    print(X_sample.iloc[idx][['Age_Oldest_TL','enq_L3m','num_std_12mts','Tot_Missed_Pmnt']])

# ---- Dependence plot: enq_L3m (2nd strongest feature) ----
enq_l3m_idx = feature_names.index('enq_L3m')
shap.dependence_plot(enq_l3m_idx, shap_values[:, :, 3], X_sample_t,
                      feature_names=feature_names, show=False)
plt.tight_layout()
plt.savefig('dependence_enq_L3m.png', dpi=100)
plt.close()
print("Dependence plot saved for enq_L3m (class P4)")


Waterfall saved for customer index 0, predicted P4
Their key raw values:
Age_Oldest_TL      51.0
enq_L3m             4.0
num_std_12mts         0
Tot_Missed_Pmnt       0
Name: 0, dtype: object
Dependence plot saved for enq_L3m (class P4)


In [1]:
import sys
print(sys.executable)

c:\Users\DURVA\Finance Project\.venv\Scripts\python.exe


In [ ]:
import sys
import sklearn
print(sys.executable)
print(sklearn.__version__)

ImportError: The `scipy` install you are using seems to be broken, (extension modules cannot be imported), please try reinstalling.

: 